# RAG Vector Benchmark - Interactive Explorer

This notebook lets you:
1. Connect to your Pinecone index and see what's stored
2. Query the database with natural language
3. Understand how embeddings map back to text/docs/images
4. Test different retrieval scenarios

## How Pinecone Works (No Docker Needed!)

```
┌─────────────────────────────────────────────────────────────────┐
│                     YOUR LOCAL MACHINE                          │
│                                                                 │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐         │
│  │ Your Python │───▶│  Pinecone   │───▶│   HTTPS     │─────────┼──▶ Pinecone Cloud
│  │   Script    │    │   Client    │    │  Requests   │         │    (stores vectors)
│  └─────────────┘    └─────────────┘    └─────────────┘         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

Pinecone = Managed cloud database (like Firebase, but for vectors)
- No installation needed
- No Docker container
- Just API calls over the internet
- Free tier: 100K vectors, 1 index
```

## 1. Setup & Connect

In [ ]:
# Add project to path
import sys
sys.path.insert(0, '.')

# Import our modules
import config
from pinecone_client import PineconeClient
from embeddings import EmbeddingManager
from rag_pipeline import RAGPipeline

print("Modules loaded successfully!")

In [ ]:
# Connect to Pinecone
client = PineconeClient()
client.connect()

# Get index statistics
stats = client.get_stats()
print("\n" + "="*50)
print("PINECONE INDEX STATUS")
print("="*50)
print(f"Index Name: {config.PINECONE_INDEX_NAME}")
print(f"Total Vectors: {stats['total_vectors']}")
print(f"Dimension: {stats['dimension']}")
print(f"Index Fullness: {stats.get('index_fullness', 'N/A')}")
print("="*50)

## 2. Explore What's Stored in Pinecone

**Key Insight:** Pinecone stores:
- `id`: Unique identifier (e.g., "wiki_0042", "arxiv_0015", "img_0001")
- `values`: The embedding vector (384 floats)
- `metadata`: Original content + source info (this is how we get text back!)

In [ ]:
# Fetch some random vectors to see what's stored
# We'll query with a random vector to get samples

import numpy as np

# Create a random query vector (just to fetch some results)
random_vector = np.random.randn(config.EMBEDDING_DIMENSION).tolist()

# Query to get samples
results = client.query(random_vector, top_k=5, include_metadata=True)

print("\n" + "="*60)
print("SAMPLE VECTORS FROM YOUR INDEX")
print("="*60)

for i, match in enumerate(results, 1):
    print(f"\n--- Vector {i} ---")
    print(f"ID: {match['id']}")
    print(f"Score: {match['score']:.4f}")
    print(f"Type: {match['metadata'].get('type', 'unknown')}")
    print(f"Topic/Source: {match['metadata'].get('topic', match['metadata'].get('source', 'N/A'))}")
    
    # Show content preview
    content = match['metadata'].get('content', match['metadata'].get('description', ''))
    if content:
        print(f"Content Preview: {content[:200]}...")

In [ ]:
# Count vectors by type
print("\n" + "="*50)
print("BREAKDOWN BY DATA TYPE")
print("="*50)

# Query for each type
for data_type in ['text', 'document_chunk', 'image']:
    try:
        # Use filter to count each type
        type_results = client.query(
            random_vector, 
            top_k=1, 
            filter={"type": data_type},
            include_metadata=True
        )
        if type_results:
            print(f"  {data_type}: Found (at least 1)")
        else:
            print(f"  {data_type}: None found")
    except Exception as e:
        print(f"  {data_type}: Query failed - {e}")

## 3. Query with Natural Language

**How it works:**
1. Your query text → Embedding model → 384-dim vector
2. Vector sent to Pinecone → Find similar vectors (cosine similarity)
3. Return top-K matches with their metadata (original text!)

In [ ]:
# Initialize embedding manager for queries
embedder = EmbeddingManager()

def search(query: str, top_k: int = 5, filter_type: str = None):
    """Search the Pinecone index with a natural language query."""
    print(f"\n🔍 Query: '{query}'")
    print("-" * 60)
    
    # Convert query to embedding
    query_embedding = embedder.embed_query(query)
    
    # Build filter if specified
    metadata_filter = {"type": filter_type} if filter_type else None
    
    # Query Pinecone
    results = client.query(
        query_embedding.tolist(),
        top_k=top_k,
        filter=metadata_filter,
        include_metadata=True
    )
    
    if not results:
        print("No results found.")
        return []
    
    print(f"Found {len(results)} results:\n")
    
    for i, match in enumerate(results, 1):
        print(f"[{i}] Score: {match['score']:.4f} | ID: {match['id']}")
        print(f"    Type: {match['metadata'].get('type', '?')}")
        print(f"    Source: {match['metadata'].get('topic', match['metadata'].get('source', '?'))}")
        
        # Show content
        content = match['metadata'].get('content', match['metadata'].get('description', ''))
        if content:
            print(f"    Content: {content[:300]}...")
        print()
    
    return results

print("Search function ready! Use: search('your query here')")

In [ ]:
# Try some queries!

# General query
search("machine learning neural networks")

In [ ]:
# Medical/scientific query (if you loaded pubmed)
search("cancer treatment research")

In [ ]:
# Financial query (if you loaded financial dataset)
search("stock market earnings report")

In [ ]:
# Filter by type - only search text documents
search("artificial intelligence", filter_type="text")

In [ ]:
# Filter by type - only search document chunks
search("deep learning", filter_type="document_chunk")

In [ ]:
# Search for images (if you loaded COCO)
search("person with dog", filter_type="image")

## 4. Understanding Embeddings → Text Mapping

**Q: How do we get text back from embeddings?**

**A:** We don't decode embeddings! Instead:
- Original text is stored in `metadata.content`
- Embeddings are just for similarity search
- Think of it as: embeddings = index, metadata = actual data

In [ ]:
# Demonstrate the relationship between embeddings and text

print("HOW EMBEDDINGS WORK")
print("="*60)
print("""
STORAGE (what Pinecone holds):
┌─────────────────────────────────────────────────────────────┐
│  id: "wiki_0042"                                            │
│  values: [0.023, -0.156, 0.089, ... 384 floats]  ← EMBEDDING│
│  metadata: {                                                │
│    "type": "text",                                          │
│    "topic": "wikipedia",                                    │
│    "content": "Kubernetes is an open-source..." ← ORIGINAL │
│  }                                                          │
└─────────────────────────────────────────────────────────────┘

RETRIEVAL:
1. Your query: "container orchestration"
2. Query → Embedding: [0.045, -0.123, 0.067, ...]
3. Pinecone finds vectors with similar values
4. Returns metadata.content → "Kubernetes is an open-source..."

KEY POINT: Embeddings are NOT reversible!
- You cannot decode [0.023, -0.156, ...] back to text
- That's why we store original text in metadata
""")

In [ ]:
# Show a real example
print("\nREAL EXAMPLE:")
print("-" * 60)

# Get one vector by ID (if you know an ID)
sample_query = "how does machine learning work"
query_emb = embedder.embed_query(sample_query)

print(f"Query: '{sample_query}'")
print(f"\nQuery Embedding (first 10 values):")
print(f"  {query_emb[:10].tolist()}")
print(f"  ... ({len(query_emb)} total dimensions)")

# Get top result
results = client.query(query_emb.tolist(), top_k=1, include_metadata=True)

if results:
    print(f"\nTop Match:")
    print(f"  ID: {results[0]['id']}")
    print(f"  Similarity Score: {results[0]['score']:.4f}")
    print(f"  Retrieved Text: {results[0]['metadata'].get('content', 'N/A')[:300]}...")

## 5. Full RAG Pipeline (Query + LLM Generation)

In [ ]:
# Initialize the full RAG pipeline
rag = RAGPipeline()
rag.connect()

print("RAG Pipeline connected!")

In [ ]:
# RAG query WITHOUT LLM generation (just retrieval)
def rag_retrieve(query: str, top_k: int = 5):
    """Retrieve relevant documents without LLM generation."""
    result = rag.rag_query(query, top_k=top_k, generate=False)
    
    print(f"\n🔍 Query: '{query}'")
    print(f"⏱️  Query Time: {result.query_time * 1000:.2f}ms")
    print(f"📚 Retrieved {len(result.retrieved_docs)} documents:\n")
    
    for i, doc in enumerate(result.retrieved_docs, 1):
        print(f"[{i}] Score: {doc.get('score', 0):.4f}")
        print(f"    Type: {doc.get('metadata', {}).get('type', '?')}")
        content = doc.get('metadata', {}).get('content', '')
        print(f"    Content: {content[:250]}...\n")
    
    return result

# Try it
rag_retrieve("explain neural network architecture")

In [ ]:
# RAG query WITH LLM generation (requires OpenAI API key)
def rag_answer(query: str, top_k: int = 3):
    """Get an AI-generated answer using retrieved context."""
    result = rag.rag_query(query, top_k=top_k, generate=True)
    
    print(f"\n🔍 Query: '{query}'")
    print(f"⏱️  Query Time: {result.query_time * 1000:.2f}ms")
    print(f"📚 Used {len(result.retrieved_docs)} documents as context")
    
    if result.generated_response:
        print(f"\n💡 Generated Answer:")
        print("-" * 50)
        print(result.generated_response)
    else:
        print("\n⚠️  No response generated (OpenAI API key may not be set)")
    
    return result

# Try it (only works if OPENAI_API_KEY is set in .env)
# rag_answer("What is machine learning?")

## 6. Fetch Specific Vectors by ID

In [ ]:
# If you know specific IDs, you can fetch them directly
def fetch_by_ids(ids: list):
    """Fetch specific vectors by their IDs."""
    results = client.index.fetch(ids=ids)
    
    print(f"Fetched {len(results.get('vectors', {}))} vectors:")
    
    for vec_id, vec_data in results.get('vectors', {}).items():
        print(f"\nID: {vec_id}")
        print(f"  Type: {vec_data['metadata'].get('type', '?')}")
        print(f"  Content: {vec_data['metadata'].get('content', 'N/A')[:200]}...")
    
    return results

# Example: Try fetching by ID patterns from your data
# fetch_by_ids(["arxiv_0001", "pubmed_0001", "fin_0001"])

## 7. Interactive Query Box

In [ ]:
# Interactive query - change the query and re-run this cell!

QUERY = "deep learning for medical diagnosis"  # ← Change this!
TOP_K = 5
FILTER_TYPE = None  # Options: "text", "document_chunk", "image", or None for all

search(QUERY, top_k=TOP_K, filter_type=FILTER_TYPE)

## 8. Visualize Embedding Similarity

In [ ]:
import numpy as np

# Compare similarity between different queries
queries = [
    "machine learning algorithms",
    "deep neural networks",
    "stock market trading",
    "medical diagnosis treatment",
    "programming python code"
]

print("Computing embeddings for queries...")
embeddings = [embedder.embed_query(q) for q in queries]

# Compute cosine similarity matrix
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nQUERY SIMILARITY MATRIX")
print("="*70)
print(f"{'':30}", end="")
for i in range(len(queries)):
    print(f"Q{i+1:2}", end="   ")
print()

for i, q1 in enumerate(queries):
    print(f"{q1[:28]:30}", end="")
    for j, q2 in enumerate(queries):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:.2f}", end="  ")
    print()

print("\nLegend:")
for i, q in enumerate(queries):
    print(f"  Q{i+1}: {q}")

## 9. Delete Vectors (Be Careful!)

In [ ]:
# WARNING: This permanently deletes vectors!
# Uncomment only if you're sure

# Delete by ID
# client.index.delete(ids=["wiki_0001", "wiki_0002"])

# Delete by filter (delete all images)
# client.index.delete(filter={"type": "image"})

# Delete ALL vectors (nuclear option)
# client.index.delete(delete_all=True)

print("Delete commands are commented out for safety.")
print("Uncomment the one you need and re-run.")

## Quick Reference

```python
# Basic search
search("your query here")

# Search with filter
search("query", filter_type="text")  # only text documents
search("query", filter_type="document_chunk")  # only chunks
search("query", filter_type="image")  # only images

# RAG retrieval (no LLM)
rag_retrieve("your question")

# RAG with answer generation (needs OpenAI key)
rag_answer("your question")

# Check index stats
client.get_stats()
```